# System Exploration (Application)

Parsing the data and understanding it (System attribute)

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from anomaly_detection.etl.load import load_records, load_system_df

plt.style.use('ggplot')

In [ ]:
# Path to the data
project_folder = Path.cwd().parent

output_path = project_folder / "data/system_exploration/application"

evtx_path = project_folder / "data/raw/93_applog.evtx"

evtx_path

In [ ]:
records = load_records(evtx_path)

len(records)

In [ ]:
with open(project_folder / "data/processed/record.txt", "w", encoding="utf-8") as file:
    file.write(json.dumps(
        records[0],
        indent=4
    ))

records[0]

In [ ]:
system_df = load_system_df(evtx_path)

system_df.head()

In [ ]:
system_df.columns

In [ ]:
system_df.describe(include='all')

# EventID and Qualifiers

In [ ]:
events_df = pd.DataFrame(system_df.value_counts(['EventID', 'Qualifiers', 'Provider_Name']))

events_df

In [ ]:
event_descriptions = {
    (4, 49152): "Generic legacy log message: PHP interpreter startup/runtime output (warnings, extension load failures)",
    (16394, 49152): "Software Protection Platform (licensing/activation) status event",
    (16384, 16384): "SPP event, distinct subtype/rule-engine notification",
    (4098, 34305): "Warning: a GPO registry preference item failed to apply (error code in payload)",
    (64, 32768): "Warning: a certificate is about to expire or has expired (autoenrollment didn't renew in time)",
    (1000, 0): "Application crash record, faulting process/module details",
    (1001, 0): "WER report logged after a crash, correlates with a 1000 event",
    (1704, 16384): "Informational: Group Policy security settings applied successfully (routine, periodic, refreshes every ~90 min on domain members or ~16h if unchanged)",
    (100, 49152): "MariaDB service log message: not a standardized Windows EventID, specific meaning requires manual check in Event Viewer",
    (9027, 16384): "Informational: DWM registered a new session port; normally logged at logon, but repeated/rhythmic occurrences are a known indicator of active RDP brute-force attempts",
}

events_df["Description"] = event_descriptions

events_df.to_csv(output_path / "events.txt")

events_df

# Constants

In [ ]:
system_df.astype(str).nunique()

In [ ]:
constants_df = pd.DataFrame(system_df[['Keywords', 'Channel', 'Computer', 'Version', 'Opcode', 'Execution_ProcessID', 'Execution_ThreadID']].value_counts())

constants_df.to_csv(output_path / "constants.csv")

constants_df

# Full Data Events

In [ ]:
full_system_df = system_df.dropna(subset=['Version'])

full_system_df.head()

In [ ]:
full_system_df[['Version', 'Opcode', 'Execution_ProcessID', 'Execution_ThreadID', 'Provider_EventSourceName']].value_counts()